# L2 logistic regression baseline

Our target is self-reported short sleep, defined as fewer than seven hours. This first run uses `C=1` to establish a validation benchmark. Logistic regression estimates how the ten inputs are associated with the odds of short sleep and gives later models a reference for prediction performance.

## 1. Load the encoded data and define the outcome

Use `Data_preprocessing/brfss_2022_encoded.csv`, exported by the preparation
notebook. It contains 79 numerical predictors representing the ten original
characteristics, plus the outcome, survey weight, design IDs, and split label.
The preparation notebook already applies ordinal and one-hot encoding.
`enough_sleep` uses 1 for at least seven hours, so we invert it to make short
sleep the positive class.

### Predictor selection based on the training EDA

The team's `BRFSS_2022_EDA.ipynb` recommends retaining all ten characteristics
for the initial model comparison. We follow that training-set decision.

| Original characteristic | Training EDA rationale for inclusion |
|---|---|
| `age_group` | Weighted short-sleep prevalence is 42.270% at ages 35–39 and 24.260% at 80+. Age provides demographic context for the other predictors. |
| `income_group` | Weighted short sleep generally decreases across income groups, with exceptions, from 41.900% below $10k to 29.920% at $200k+. |
| `education_level` | Weighted rates differ across education groups, including 40.550% in grades 9–11 and 31.000% among college graduates. |
| `employment_status` | Weighted short sleep is 49.070% among respondents unable to work and 27.390% among retired respondents. |
| `physical_activity` | Weighted short sleep is 41.050% with no leisure-time activity and 34.520% with activity. |
| `smoking_status` | Weighted short sleep is 49.600% among daily smokers and 33.660% among never smokers. |
| `sex`, `marital_status`, `children_group`, `state` | Retain the remaining demographic, household, and geographic characteristics as context, following the EDA's explicit recommendation to retain all ten. |

These unadjusted group differences motivate joint modeling. The EDA's largest
pairwise Cramér's V is 0.336 and provides no clear basis for removing a nearly
duplicated input. We retain the supplied Unknown indicators, particularly
because income and smoking have 20.990% and 7.850% Unknown in the training set.
Outcome fields and survey-design fields are excluded from the feature matrix.

The shared export uses ordinal codes for age, income, education, and children,
with a separate Unknown indicator for each. Logistic regression therefore
assumes a constant change in log-odds per rank step among known levels.
The EDA shows non-monotonic age and education patterns, so this linearity
assumption is a limitation of using the shared ordinal representation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

cwd = Path.cwd()
data_dir = next(
    (
        folder / "Data_preprocessing"
        for folder in (cwd, *cwd.parents)
        if (folder / "Data_preprocessing" / "brfss_2022_encoded.csv").is_file()
    ),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find Data_preprocessing/brfss_2022_encoded.csv. "
        "Run the encoding section of the data preparation notebook first."
    )

# Keep the ten original characteristics recommended by the training EDA.
selected_variables = [
    "age_group", "sex", "income_group", "education_level",
    "employment_status", "marital_status", "children_group",
    "physical_activity", "smoking_status", "state",
]
df = pd.read_csv(data_dir / "brfss_2022_encoded.csv")

# Select their existing ordinal, Unknown, and one-hot columns.
features = [
    column for column in df.columns
    if any(
        column == variable or column.startswith(variable + "_")
        for variable in selected_variables
    )
]
metadata_columns = [
    "enough_sleep", "survey_weight", "stratum_id", "psu_id", "split",
]
assert set(metadata_columns).issubset(df.columns), "Missing outcome or survey fields."
assert features and set(features).isdisjoint(metadata_columns + ["sleep_hours"])
assert set(df.columns) == set(features + metadata_columns), "Unexpected encoded CSV columns."
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in features)
assert np.isfinite(df[features].to_numpy()).all(), "Encoded predictors must be finite."
assert df["enough_sleep"].isin([0, 1]).all()
assert np.isfinite(df["survey_weight"]).all() and df["survey_weight"].gt(0).all()
df["short_sleep"] = 1 - df["enough_sleep"]

print(f"Respondents: {len(df):,}")
print(f"Short-sleep respondents: {df['short_sleep'].sum():,}")
print(f"Model features from the encoded CSV: {len(features)}")

Respondents: 427,573
Short-sleep respondents: 139,673
Model features from the encoded CSV: 79


## 2. Use the existing train, validation, and test assignments

The encoded CSV includes the shared `split` column. During preprocessing, each state and primary sampling unit (PSU) combination was assigned to one partition. We use those assignments so this baseline and the tree models are compared on the same respondents. Training rows are used for fitting, validation rows are used for comparing settings, and test rows are reserved for final evaluation.

In [2]:
split_order = ["train", "validation", "test"]

if "split" not in df.columns:
    raise ValueError("No Split column found in the dataset")
assert set(df["split"]) == set(split_order), "Each respondent must have a valid split label."

counts = df["split"].value_counts().reindex(split_order)
display(counts.to_frame("respondents"))

train = df.loc[df["split"].eq("train")]
validation = df.loc[df["split"].eq("validation")]
display(
    df.loc[df["split"].isin(["train", "validation"])]
    .groupby("split")["short_sleep"].mean().mul(100).round(3)
    .to_frame("sample short sleep (%)")
)

,respondents
split,
train,256605
validation,85485
test,85483


,sample short sleep (%)
split,
train,32.623
validation,32.871


## 3. Fit an L2 logistic regression

Fit logistic regression directly on the numerical columns exported during
preprocessing. L2 regularization shrinks large coefficients. Smaller `C` means
stronger shrinkage. Training survey weights are divided by their mean so they
retain their relative weights while averaging one.

In [3]:
train_weights = train["survey_weight"]
model = LogisticRegression(C=1, solver="lbfgs", max_iter=1_000)
model.fit(
    train[features],
    train["short_sleep"],
    sample_weight=train_weights / train_weights.mean(),
)
print(f"Model features: {len(features)}")

Model features: 79


## 4. Score on the validation set

Survey-weighted average precision (AP) is our primary metric. It summarizes precision across recall levels when we rank respondents by their predicted probability of short sleep. The constant-score reference has AP equal to the validation set's weighted prevalence. ROC-AUC summarizes how well scores rank short sleepers ahead of other respondents.

In [4]:
val_weights = validation["survey_weight"]
y_val = validation["short_sleep"]
scores = model.predict_proba(validation[features])[:, 1]
weighted_prevalence = np.average(y_val, weights=val_weights)

metrics = pd.DataFrame(
    [
        [weighted_prevalence, 0.5],
        [
            average_precision_score(y_val, scores, sample_weight=val_weights),
            roc_auc_score(y_val, scores, sample_weight=val_weights),
        ],
    ],
    index=["Constant score", "L2 logistic regression"],
    columns=["Weighted AP", "Weighted ROC-AUC"],
)
display(metrics.round(3))

,Weighted AP,Weighted ROC-AUC
Constant score,0.363,0.500
L2 logistic regression,0.454,0.602


## 5. Interpret selected category contrasts

Let $p$ be the model's predicted probability of short sleep (less than seven hours).
Logistic regression models the log-odds using the exported numerical predictors:

$$
\log\!\left(\frac{p}{1-p}\right)
= \beta_0 + \sum_{j=1}^{J}\beta_j x_j,
$$

where $J$ is the number of selected encoded columns. With the other predictors
fixed, changing one encoded value by $\Delta x$ gives

$$
\mathrm{OR}=\exp(\beta_j\Delta x).
$$

The preparation notebook omits the one-hot reference categories. For activity,
code 1 (any activity) is the reference, so comparing code 2 (no activity) with
code 1 changes `physical_activity_2` from 0 to 1. For employment, code 1
(employed for wages) is the reference, so comparing code 8 (unable to work)
with code 1 changes `employment_status_8` from 0 to 1.

Income is ordinal in the shared CSV. Comparing code 1 (under $10k) with code 7
($50k to under $75k) changes the encoded income by $1-7=-6$, giving
$\exp(-6\beta_{\text{income}})$. Both categories are known, so the income
Unknown indicator stays zero. This comparison inherits the model's linear
log-odds assumption across known income ranks.

An odds ratio above 1 means higher modeled odds of short sleep for the
comparison category, holding the other predictors fixed. These ratios
describe associations in the fitted model.

In [5]:
coefficients = pd.Series(model.coef_[0], index=features)

# Each row specifies the encoded column and its change between the two groups.
contrasts = [
    ("physical_activity_2", 1, "No leisure-time activity vs. any"),
    ("income_group", -6, "Household income under $10k vs. $50k to under $75k"),
    ("employment_status_8", 1, "Unable to work vs. employed for wages"),
]
odds_ratios = pd.DataFrame(
    [
        {
            "Contrast": description,
            "Odds ratio": np.exp(coefficients[column] * change),
        }
        for column, change, description in contrasts
    ]
)
display(odds_ratios.round(3))

,Contrast,Odds ratio
0,No leisure-time activity vs. any,1.237
1,Household income under $10k vs. $50k to under ...,1.208
2,Unable to work vs. employed for wages,1.226


## 6. Tune L2 regularization on the validation set

We randomly sample 20 values of `C` on a log scale from 0.001 to 100. We fit each candidate on the same training set and select the setting with the highest survey-weighted average precision on the validation set. We also report weighted ROC-AUC and compare each candidate with the original `C=1` baseline. The test set is reserved for final evaluation.

In [6]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler

# Search across orders of magnitude. A fixed seed makes the draw reproducible.
c_samples = ParameterSampler(
    {"C": loguniform(1e-3, 100)},
    n_iter=20,
    random_state=42,
)

# Keep the original C=1 result as a reference.
baseline_ap = metrics.loc["L2 logistic regression", "Weighted AP"]
baseline_auc = metrics.loc["L2 logistic regression", "Weighted ROC-AUC"]
results = [
    {"C": 1, "Weighted AP": baseline_ap, "Weighted ROC-AUC": baseline_auc}
]

best_c = 1
best_ap = baseline_ap
best_model = model
normalized_weights = train_weights / train_weights.mean()

for params in c_samples:
    c = params["C"]

    # Start with a fresh copy of the classifier and change only C.
    candidate = clone(model).set_params(C=c)
    candidate.fit(
        train[features],
        train["short_sleep"],
        sample_weight=normalized_weights,
    )

    # Score every candidate on the same validation rows with the same weights.
    candidate_scores = candidate.predict_proba(validation[features])[:, 1]
    ap = average_precision_score(y_val, candidate_scores, sample_weight=val_weights)
    auc = roc_auc_score(y_val, candidate_scores, sample_weight=val_weights)
    results.append({"C": c, "Weighted AP": ap, "Weighted ROC-AUC": auc})

    if ap > best_ap:
        best_c = c
        best_ap = ap
        best_model = candidate

# Rank by the full-precision AP and round only for display.
tuning_results = pd.DataFrame(results).sort_values(
    "Weighted AP", ascending=False, kind="stable"
).reset_index(drop=True)

tuning_results["AP gain (percentage points)"] = 100 * (
    tuning_results["Weighted AP"] - baseline_ap
)

display(tuning_results.round(3))
print(f"Best C: {best_c:.3f}")
print(f"Best validation weighted AP: {best_ap:.3f}")
print(f"AP gain vs. C=1: {100 * (best_ap - baseline_ap):.3f} percentage points")

,C,Weighted AP,Weighted ROC-AUC,AP gain (percentage points)
0,0.008,0.454,0.602,0.010
1,0.008,0.454,0.602,0.009
2,0.012,0.454,0.602,0.008
3,0.029,0.454,0.602,0.005
4,0.006,0.454,0.602,0.004
5,0.006,0.454,0.602,0.004
6,0.033,0.454,0.602,0.004
7,0.421,0.454,0.602,0.002
8,14.528,0.454,0.602,0.002
9,70.721,0.454,0.602,0.001


Best C: 0.008
Best validation weighted AP: 0.454
AP gain vs. C=1: 0.010 percentage points
